# E10c · Otras formulaciones del problema y modelos base de familias distintas
**Parte de:** E8b (bagging de 5 semillas con los 5 modelos originales + LightGBM, CV 0,8179, Kaggle público 0,80168).

**Hipótesis C.** Lo único que mejoró el stacking después de E3 fue agregar el único modelo **no lineal** (LGBM) y promediar semillas. Si se agregan modelos que **formulan el problema de otra manera** o que pertenecen a **otra familia** (árboles por *bagging*, vecinos, Naive Bayes de presencia), sus errores deberían estar menos correlacionados con los de las LR y el metamodelo tendría más que combinar.

Modelos nuevos (código en `ext_formulaciones.py`; no se toca `toxic_base.py`):

| Modelo | Familia / formulación | Representación |
|---|---|---|
| **Jerarq-LR** | Condicional: HS con LR en todo el train; TR y AG con LR entrenadas **solo con los tweets HS=1**; AG para HS=0 con una LR aparte (los 26 casos `001`, casi siempre comillas). P(TR)=P(HS)·P(TR\|HS), P(AG)=P(HS)·P(AG\|HS)+(1−P(HS))·P(AG\|no HS) | word + char + léxico |
| **ExtraTrees** | Árboles aleatorizados por *bagging* (no *boosting*) | SVD(300) + léxico crudo |
| **kNN** | 25 vecinos más cercanos por similitud coseno, voto ponderado | TF-IDF word + char |
| **BNB** | Bernoulli Naive Bayes sobre presencia/ausencia de palabras | 1-2 gramas de palabras binarizados |
| **LGBM-PS** | LightGBM *Label Powerset*: multiclase sobre las 6 combinaciones HS-TR-AG | SVD(300) + léxico crudo |
| **LGBM-chi2** | LightGBM sobre TF-IDF **disperso** filtrado con chi² (2000 n-gramas por etiqueta, elegidos solo con la partición de entrenamiento) | word + char + léxico crudo |

**Protocolo.** (1) Cada modelo solo con CV repetida de 3 semillas (42, 43, 44). (2) Correlación de sus OOF con LR y LGBM. (3) Filtro rápido: stacking simple (una semilla de bases) de B6 + el modelo. (4) Los que no empeoran pasan al protocolo de E8b: OOF y predicciones de test por partición con las 5 semillas (42-46) y `stacking_bagged` con conjuntos **definidos de antemano**, comparados con E8b (0,8179). Criterio de adopción de `RESUMEN.md`: +0,003 y mejora en al menos 2 de 3 semillas.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking / `stacking_bagged`** | Un metamodelo (LR) combina las probabilidades OOF; en la versión *bagged* cada base es el promedio de 5 semillas |
| **B6** | Los 5 modelos originales (CNB, LR, SVM, NB-LR, Powerset) + LGBM: las bases de E8b |
| **Formulación jerárquica / condicional** | Modelar TR y AG como "dado que hay odio": P(TR) = P(HS)·P(TR \| HS=1) |
| **ExtraTrees** | *Extremely Randomized Trees*: muchos árboles con cortes al azar, promediados (familia *bagging*) |
| **kNN** | *k-Nearest Neighbors*: predice con las etiquetas de los k tweets más parecidos del train |
| **Similitud coseno** | Coseno del ángulo entre dos vectores TF-IDF: 1 = mismas palabras en igual proporción |
| **Bernoulli NB** | Naive Bayes que solo mira si cada palabra aparece o no (no cuántas veces) |
| **Label Powerset** | Convierte las 3 etiquetas en una sola clase combinada (`000`, `111`, `101`...) |
| **chi² (chi cuadrado)** | Prueba estadística que mide cuánto se asocia cada n-grama con la etiqueta; se usa para quedarse con los 2000 más informativos |
| **Correlación de Pearson** | Entre −1 y 1: cuán parecidas son las probabilidades de dos modelos. Cerca de 1 = aportan lo mismo |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
os.environ.setdefault('OMP_NUM_THREADS', '3')
import numpy as np, pandas as pd, sklearn, scipy, lightgbm
import toxic_base as tb
import ext_formulaciones as ef                     # registra los modelos nuevos en tb.MODELOS
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__, '| lightgbm', lightgbm.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)
print('Modelos nuevos:', ef.NUEVOS)
print('Combinaciones en el train:', pd.Series(d.COMBO).value_counts().to_dict())

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1 | lightgbm 4.7.0
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)
Modelos nuevos: ['Jerarq-LR', 'ExtraTrees', 'kNN', 'BNB', 'LGBM-PS', 'LGBM-chi2']
Combinaciones en el train: {'000': 949, '111': 491, '101': 145, '100': 92, '110': 28, '001': 26}


## 1. Modelos nuevos por separado (CV repetida, 3 semillas)

In [2]:
B6 = tb.BASES_ORIGINALES + ['LGBM']
R = {}
for b in B6 + ef.NUEVOS:
    R[b] = tb.evaluar(d, {s: tb.oof(d, b, s) for s in tb.SEEDS})
tb.log('OOF listos')
tb.tabla(R)

[     1s] OOF listos


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
CNB,0.7509,0.7794,0.7086,0.7463,0.0023,0.7090,0.749 / 0.744 / 0.746
LR,0.7924,0.8246,0.7899,0.8023,0.0020,0.7499,0.803 / 0.805 / 0.800
SVM,0.7890,0.8279,0.7882,0.8017,0.0009,0.7527,0.803 / 0.801 / 0.801
NB-LR,0.7916,0.8231,0.7883,0.8010,0.0009,0.7578,0.801 / 0.802 / 0.800
Powerset,0.7937,0.8284,0.7889,0.8037,0.0021,0.7793,0.806 / 0.804 / 0.801
LGBM,0.7858,0.8182,0.7743,0.7928,0.0007,0.7410,0.792 / 0.793 / 0.793
Jerarq-LR,0.7924,0.8223,0.7810,0.7985,0.0010,0.7531,0.800 / 0.798 / 0.797
ExtraTrees,0.7744,0.8093,0.7816,0.7885,0.0026,0.7551,0.787 / 0.786 / 0.792
kNN,0.7440,0.7854,0.6940,0.7411,0.0024,0.7048,0.744 / 0.741 / 0.738


## 2. ¿Aportan diversidad? Correlación de las OOF con LR, Powerset y LGBM
Probabilidades OOF promediadas en las 3 semillas. Buscamos correlación baja **y** un F1 razonable.

In [3]:
filas = []
for j, l in enumerate(tb.LABELS):
    df = pd.DataFrame({b: np.mean([tb.oof(d, b, s)[:, j] for s in tb.SEEDS], 0) for b in B6 + ef.NUEVOS}).corr()
    for b in ef.NUEVOS:
        filas.append({'Modelo': b, 'Etiqueta': l, 'corr LR': df.loc[b, 'LR'], 'corr Powerset': df.loc[b, 'Powerset'],
                      'corr LGBM': df.loc[b, 'LGBM']})
CORR = pd.DataFrame(filas).pivot(index='Modelo', columns='Etiqueta').round(3)
CORR = CORR.reindex(ef.NUEVOS)
CORR['corr LR medio'] = CORR['corr LR'].mean(1).round(3)
CORR['F1 solo'] = [round(R[b]['F1 medio'], 4) for b in CORR.index]
CORR

corr LR               corr Powerset               corr LGBM               corr LR medio F1 solo
Etiqueta        AG     HS     TR            AG     HS     TR        AG     HS     TR                      
Modelo                                                                                                    
Jerarq-LR    0.981  1.000  0.984         0.979  0.977  0.988     0.945  0.929  0.960         0.988  0.7985
ExtraTrees   0.929  0.917  0.959         0.925  0.919  0.942     0.975  0.958  0.986         0.935  0.7885
kNN          0.775  0.807  0.876         0.782  0.809  0.870     0.818  0.846  0.867         0.819  0.7411
BNB          0.797  0.820  0.872         0.793  0.807  0.859     0.803  0.820  0.854         0.830  0.7373
LGBM-PS      0.925  0.916  0.971         0.930  0.922  0.960     0.981  0.978  0.994         0.937  0.7915
LGBM-chi2    0.912  0.901  0.953         0.921  0.899  0.949     0.930  0.920  0.955         0.922  0.7837

Referencia: entre los modelos de B6 (HS), LR–Powerset ≈ 0,98 y LR–LGBM ≈ 0,93 (ver E3).

## 3. Filtro rápido: stacking simple de B6 + cada modelo
Stacking de `tb.stacking` (una semilla de bases, la misma de la CV del metamodelo). Sirve solo para descartar los que restan.

In [4]:
def ev_simple(bs):
    return tb.evaluar(d, {s: tb.stacking(d, bs, s, con_test=False)[0] for s in tb.SEEDS})
RF = {'B6 (stacking simple)': ev_simple(B6)}
for b in ef.NUEVOS:
    RF['B6 + ' + b] = ev_simple(B6 + [b])
tb.tabla(RF)

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
B6 (stacking simple),0.8072,0.8336,0.7982,0.8130,0.0013,0.7737,0.814 / 0.813 / 0.811
B6 + Jerarq-LR,0.8079,0.8340,0.7999,0.8139,0.0010,0.7753,0.813 / 0.815 / 0.813
B6 + ExtraTrees,0.8081,0.8364,0.7990,0.8145,0.0014,0.7772,0.815 / 0.816 / 0.813
B6 + kNN,0.8047,0.8329,0.7981,0.8119,0.0007,0.7714,0.813 / 0.812 / 0.811
B6 + BNB,0.8064,0.8364,0.7977,0.8135,0.0012,0.7749,0.815 / 0.812 / 0.813
B6 + LGBM-PS,0.8111,0.8346,0.7995,0.8151,0.0033,0.7776,0.818 / 0.816 / 0.811
B6 + LGBM-chi2,0.8082,0.8323,0.8011,0.8139,0.0030,0.7762,0.814 / 0.818 / 0.810


kNN es el modelo menos correlacionado (≈ 0,8), pero es tan débil (0,741 solo) que el metamodelo no le saca provecho y el stacking baja. Se descarta y no se calcula con 5 semillas. Los otros cinco pasan a la prueba con *bagging*.

## 4. Protocolo E8b: bagging de 5 semillas
Se calculan los OOF de las semillas 45 y 46 y las predicciones de test por partición (`pred_test_folds`) de las 5 semillas, para que queden en caché (`oof/base/`). El test solo se transforma: nunca entra en un ajuste.

In [5]:
S5 = (42, 43, 44, 45, 46)
PASAN = ['Jerarq-LR', 'ExtraTrees', 'BNB', 'LGBM-PS', 'LGBM-chi2']
for b in B6 + PASAN:
    for s in S5:
        tb.oof(d, b, s); tb.pred_test_folds(d, b, s)
    tb.log('5 semillas + test por partición:', b)

[     4s] 5 semillas + test por partición: CNB


[     4s] 5 semillas + test por partición: LR


[     4s] 5 semillas + test por partición: SVM


[     4s] 5 semillas + test por partición: NB-LR


[     4s] 5 semillas + test por partición: Powerset


...

[    36s] 5 semillas + test por partición: Jerarq-LR


[   418s] 5 semillas + test por partición: ExtraTrees


[   419s] 5 semillas + test por partición: BNB


[   644s] 5 semillas + test por partición: LGBM-PS


[   790s] 5 semillas + test por partición: LGBM-chi2


Conjuntos definidos de antemano (no búsqueda exhaustiva). La semilla del metamodelo recorre `tb.SEEDS`, igual que en E8b.

In [6]:
ARBOLES = ['ExtraTrees', 'LGBM-PS', 'LGBM-chi2']
CONJ = {'E8b · B6': B6}
for b in PASAN:
    CONJ['B6 + ' + b] = B6 + [b]
CONJ['B6 + ExtraTrees + LGBM-PS'] = B6 + ['ExtraTrees', 'LGBM-PS']
CONJ['B6 + 3 árboles (ET, LGBM-PS, LGBM-chi2)'] = B6 + ARBOLES
CONJ['B6 + los 5 nuevos'] = B6 + PASAN
CONJ['Sin CNB + 3 árboles'] = ['LR', 'SVM', 'NB-LR', 'Powerset', 'LGBM'] + ARBOLES
RB, POOF = {}, {}
for k, bs in CONJ.items():
    POOF[k] = {s: tb.stacking_bagged(d, bs, S5, s, con_test=False)[0] for s in tb.SEEDS}
    RB[k] = tb.evaluar(d, POOF[k])
TB = tb.tabla(RB)
TB['Δ vs E8b'] = (TB['F1 medio'] - TB.loc['E8b · B6', 'F1 medio']).round(4)
f1_sem = lambda k: [tb.metricas(d.Y, tb.binarizar(POOF[k][s]))['F1 medio'] for s in tb.SEEDS]
TB['semillas que mejoran'] = [sum(a > b + 1e-9 for a, b in zip(f1_sem(k), f1_sem('E8b · B6'))) for k in TB.index]
TB.sort_values('F1 medio', ascending=False)

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla,Δ vs E8b,semillas que mejoran
Configuración,,,,,,,,,
B6 + LGBM-PS,0.8123,0.8400,0.8088,0.8204,0.0004,0.7822,0.820 / 0.820 / 0.821,0.0025,3
B6 + ExtraTrees + LGBM-PS,0.8120,0.8409,0.8075,0.8201,0.0005,0.7824,0.820 / 0.821 / 0.820,0.0022,3
B6 + Jerarq-LR,0.8082,0.8389,0.8091,0.8187,0.0004,0.7789,0.819 / 0.818 / 0.818,0.0008,3
E8b · B6,0.8071,0.8405,0.8062,0.8179,0.0012,0.7785,0.819 / 0.818 / 0.816,0.0000,0
B6 + BNB,0.8073,0.8391,0.8041,0.8168,0.0007,0.7789,0.818 / 0.816 / 0.816,-0.0011,1
B6 + los 5 nuevos,0.8093,0.8351,0.8054,0.8166,0.0002,0.7803,0.817 / 0.816 / 0.817,-0.0013,1
Sin CNB + 3 árboles,0.8103,0.8364,0.8029,0.8166,0.0010,0.7812,0.816 / 0.816 / 0.818,-0.0013,1
B6 + LGBM-chi2,0.8084,0.8356,0.8046,0.8162,0.0030,0.7789,0.818 / 0.812 / 0.818,-0.0017,1
B6 + ExtraTrees,0.8071,0.8395,0.8013,0.8159,0.0012,0.7764,0.816 / 0.814 / 0.817,-0.0020,1


## 5. ¿Cuánto cambia la predicción del test?
Filas del test (con umbral 0,5) que cambian frente a E8b para el mejor conjunto. Solo se mide; **no** se genera submission.

In [7]:
mejor = TB.drop('E8b · B6')['F1 medio'].idxmax()
_, Pt8b = tb.stacking_bagged(d, B6, S5, 42)
_, Ptm = tb.stacking_bagged(d, CONJ[mejor], S5, 42)
y8b, ym = tb.binarizar(Pt8b), tb.binarizar(Ptm)
y8b[d.err_excel_te] = 0; ym[d.err_excel_te] = 0
print('Mejor conjunto:', mejor, '| bases:', CONJ[mejor])
print('Filas del test distintas frente a E8b:', int((y8b != ym).any(1).sum()), 'de', len(ym))
print('Prevalencia E8b :', dict(zip(tb.LABELS, y8b.mean(0).round(3))))
print('Prevalencia mejor:', dict(zip(tb.LABELS, ym.mean(0).round(3))))

Mejor conjunto: B6 + LGBM-PS | bases: ['CNB', 'LR', 'SVM', 'NB-LR', 'Powerset', 'LGBM', 'LGBM-PS']
Filas del test distintas frente a E8b: 25 de 1154
Prevalencia E8b : {'HS': np.float64(0.403), 'TR': np.float64(0.299), 'AG': np.float64(0.369)}
Prevalencia mejor: {'HS': np.float64(0.408), 'TR': np.float64(0.305), 'AG': np.float64(0.368)}


## Conclusión: **candidato, pero no cumple el criterio de adopción** (B6 + LGBM-PS: +0,0025, 3/3 semillas)

**Modelos solos.** Ningún modelo nuevo supera a los lineales (0,80). El más fuerte es Jerarq-LR (0,7985), seguido de LGBM-PS (0,7915) y ExtraTrees (0,7885); kNN (0,741) y BNB (0,737) quedan al nivel de CNB.

**Diversidad.** Hay dos grupos.
- kNN y BNB son los menos correlacionados con LR (≈ 0,82-0,83), pero son tan débiles que el metamodelo no les saca provecho. kNN resta en el filtro rápido y BNB resta con *bagging* (−0,0011).
- Jerarq-LR comparte el modelo de HS con LR (correlación 1,000) y en TR/AG llega a 0,98. Es casi la misma información; la suma es mínima (+0,0008).
- Los árboles nuevos se correlacionan ≈ 0,92-0,94 con LR, pero **≈ 0,98 con LGBM** (LGBM-PS) o 0,96-0,98 (ExtraTrees), porque ven las mismas 300 dimensiones de SVD.

**Stacking con *bagging* (protocolo E8b, 5 semillas).**
- El único que suma de forma consistente es **LGBM-PS**: 0,8204 ± 0,0004 frente a 0,8179 (+0,0025), mejora en las 3 semillas del metamodelo (0,820 / 0,820 / 0,821) y baja la desviación. Su aporte está en HS (+0,005) y AG (+0,003). Es decir, sirve la **formulación conjunta** (Powerset), no la familia de árboles: ExtraTrees solo resta (−0,0020) y LGBM-chi2 es inestable (0,812-0,818).
- Agregar varios árboles a la vez **empeora** (3 árboles: −0,0027; los 5 nuevos: −0,0013). El metamodelo LR no regularizado reparte el peso entre bases casi colineales y sobreajusta.

**Lectura honesta.**
- +0,0025 queda por debajo del umbral de +0,003 de `RESUMEN.md`.
- Además, es el mejor de 9 conjuntos evaluados, así que parte de la ganancia puede ser selección. Igual, que mejore en 3/3 semillas con una desviación de 0,0004 lo hace el resultado más sólido de esta prueba.
- Cambia 25 de 1154 filas del test frente a E8b, con prevalencias casi iguales. En el leaderboard público (±2,5 puntos de ruido) la diferencia no se podría distinguir.

**Decisión.**
- No se adopta como nuevo modelo de referencia.
- B6 + LGBM-PS queda registrado como **candidato** a envío o a combinarse con otras hipótesis (por ejemplo, umbral 0,45 o meta C=0,3), con OOF y test por partición de las 5 semillas ya en caché.
- La hipótesis C, en general, **no se confirma**: otras familias (ExtraTrees, kNN, Naive Bayes Bernoulli) y la formulación jerárquica no aportan al stacking.
- No se genera submission.